# Amazon Bedrock AgentCore Policy - 시작하기 데모

## 개요

Amazon Bedrock AgentCore Policy 실습 데모에 오신 것을 환영합니다! 이 Notebook에서는 AI Agent와 도구 간 상호 작용을 정책에 따라 결정론적으로 제어하도록 설정하고 테스트하는 전체 워크플로를 살펴봅니다.

### AgentCore Policy란?

Amazon Bedrock AgentCore Policy는 Agent 작업 주위에 보호 경계("안전 상자")를 만들어 AI Agent와 도구 간 상호 작용에 대한 보안 제어를 정의하고 적용할 수 있게 합니다. AI Agent는 복잡한 문제를 해결하도록 동적으로 적응할 수 있지만, 이러한 유연성으로 인해 다음과 같은 보안 문제가 발생할 수 있습니다.

- **데이터 유출**: Agent가 의도치 않게 비공개 정보를 노출할 수 있습니다.
- **비즈니스 규칙 위반**: Agent가 비즈니스 규칙을 잘못 해석하거나 우회할 수 있습니다.
- **권한 남용**: Agent가 의도된 범위를 벗어나 동작할 수 있습니다.

Policy는 AgentCore Gateway를 통해 유입되는 Agent 트래픽을 가로채고, 도구 액세스를 허용하기 전에 각 요청을 정의된 정책에 따라 평가합니다.

### 주요 이점

✅ **선언적 보안**: 코드가 아닌 Cedar 언어로 정책을 정의합니다.  
✅ **런타임 적용**: 정책을 실시간으로 평가합니다.  
✅ **세분화된 제어**: 광범위한 제한부터 세밀한 액세스 제어 및 권한 부여까지 지원합니다.
✅ **관심사 분리**: 보안 로직을 Agent 코드 외부에 둡니다.  
✅ **엔터프라이즈 규모**: 프로덕션 환경에 자율 Agent를 안전하게 배포합니다.  

---

## 데모 아키텍처

```
┌─────────────┐
│   AI Agent  │
└──────┬──────┘
       │
       │ 도구 호출 요청
       ▼
┌─────────────────────┐
│  AgentCore Gateway  │
│  + OAuth 인증       │
└──────┬──────────────┘
       │
       │ Policy 확인
       ▼
┌─────────────────────┐
│   Policy Engine     │
│   (Cedar 정책)      │
└──────┬──────────────┘
       │
       │ ALLOW / DENY
       ▼
┌─────────────────────--┐
│   Gateway 대상        │
│                       │    
└─────────────────────--┘
```

---

## 학습 내용

이 데모에서는 보험 인수 심사를 지원하는 도구와 Agent를 설정합니다.

1. **인프라 설정**: 보험 신청 생성, 위험 모델 호출, 보험 청구 승인을 위한 Lambda 대상을 포함하는 Gateway를 생성합니다.
2. **Policy Engine 생성**: Gateway용 Policy Engine을 초기화합니다.
3. **정책 정의**: 액세스를 제어하는 Cedar 정책을 작성합니다.
4. **정책 적용 테스트**: 실제 Agent 요청으로 정책이 작동하는지 확인합니다.
5. **결과 이해**: ALLOW 및 DENY 시나리오를 해석합니다.

---

## 사전 요구 사항

시작하기 전에 다음 사항을 확인하세요.

- ✅ 적절한 자격 증명으로 구성된 AWS CLI
- ✅ boto3가 설치된 Python 3.10 이상
- ✅ 설치된 `bedrock_agentcore_starter_toolkit` 패키지
- ✅ AWS Lambda 액세스 권한(대상 함수용)

---

## 데모 시나리오: 보험 인수 심사 처리

정책 제어가 적용된 **보험 인수 심사 처리 시스템**을 구현합니다.

- **도구**: 
       - `ApplicationTool` - 리전과 `coverage_amount`를 지정하여 보험 신청을 생성합니다.
       - `RiskModelTool` - API 분류 및 데이터 거버넌스 승인이 제공되면 거버넌스 제어를 적용하여 외부 위험 점수 모델을 호출합니다.
       - `ApprovalTool` - `claim_amount`와 `risk_level`을 기반으로 고액 또는 고위험 인수 결정을 승인합니다.


시작해 보겠습니다! 🚀

---

# Step 0: 환경 설정

먼저 환경을 확인하고 필요한 라이브러리를 가져옵니다.

In [ ]:
# 필수 패키지 설치
%pip install -r requirements.txt

In [ ]:
# 업데이트를 적용하기 위해 커널 재시작
import IPython

IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# 필수 라이브러리 가져오기
import sys
from pathlib import Path
import boto3
import json
import logging

In [ ]:
# Python 경로에 scripts 디렉터리 추가
scripts_dir = Path.cwd() / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# 리전 입력 요청
session = boto3.Session()
region = session.region_name
if not region:
    region = input("Enter AWS region (e.g., us-east-1, us-west-2): ").strip()
    if not region:
        raise ValueError("AWS region is required")

print(f"Region: {region}")

# AWS 자격 증명 확인
try:
    sts = session.client("sts", region_name=region)
    identity = sts.get_caller_identity()
    print("✅ AWS Credentials Verified")
    print(f"   Account: {identity['Account']}")
    print(f"   User/Role: {identity['Arn']}")
except Exception as e:
    print(f"❌ AWS Credentials Error: {e}")
    print("   Please configure AWS CLI with: aws configure")

---

# Step 1: AgentCore Gateway용 대상 함수 생성

Gateway를 설정하기 전에 Agent의 도구 대상으로 사용할 함수를 준비해야 합니다.

## Lambda 대상이란?

Lambda 대상은 AI Agent가 Gateway를 통해 호출할 수 있는 백엔드 함수입니다. 여기서는 Agent가 보험 인수 심사 작업을 수행하도록 지원하기 위해 `ApplicationTool`, `RiskModelTool`, `ApprovalTool`용 Lambda 함수 3개를 설정합니다. 

### CLI로 생성(아래 셀 실행)
다음 스크립트를 실행하면 AWS 계정에 Lambda 함수 3개가 배포됩니다.

1. Application Tool: 간소화된 신청 생성(데모용 모의 구현)
 신청자의 리전과 보장 금액을 사용하여 보험 신청을 생성합니다.
 파라미터:
 - applicant_region: 고객의 지리적 리전
 - coverage_amount: 요청한 보험 보장 금액

2. Risk Model Tool: 간소화된 위험 모델 액세스(데모용 모의 구현)
 위험 점수 모델을 호출하고 평가 결과를 반환합니다.
 파라미터:
 - API_classification: API 분류(`public`, `internal`, `restricted`)
 - data_governance_approval: 데이터 거버넌스에서 모델 사용을 승인했는지 여부

3. Approval Tool: 보험 승인 절차(데모용 모의 구현)
 인수 결정과 청구 금액을 승인합니다.
 파라미터:
 - claim_amount: 보험 청구/보장 금액
 - risk_level: 위험 수준 평가(`low`, `medium`, `high`, `critical`)

In [ ]:
%run scripts/lambda-target-setup/deploy_lambdas.py --region $region

---

# Step 2: AgentCore Gateway 설정

이제 OAuth 인증을 사용하는 AgentCore Gateway를 생성하고 Lambda 함수를 대상으로 연결합니다.

## 생성되는 리소스

1. **OAuth 권한 부여 서버**: 인증을 위한 Cognito 기반 OAuth
2. **AgentCore Gateway**: MCP 프로토콜을 지원하는 기본 Gateway
3. **Lambda 대상**: 스키마 정의와 함께 연결되는 Lambda 함수
4. **구성 파일**: 나중에 사용할 수 있도록 저장되는 모든 연결 정보

## Gateway 구성

Gateway는 다음과 같이 구성됩니다.
- **프로토콜**: MCP (Model Context Protocol)
- **인증**: Cognito를 통한 OAuth 2.0
- **대상**: `ApplicationTool`, `RiskModelTool`, `ApprovalTool` Lambda 함수 - 대상 스키마가 Gateway에 제공됩니다.

In [ ]:
# Gateway 설정 스크립트 실행
print("🚀 Setting up AgentCore Gateway...\n")
print("This will:")
print("  1. Create OAuth authorization server (Cognito)")
print("  2. Create AgentCore Gateway")
print("  3. Attach Lambda as target")
print("  4. Save configuration to config.json")
print("\n" + "=" * 60)

# Gateway 설정 스크립트 실행
%run scripts/setup_gateway.py

### Gateway 구성 확인

방금 생성한 Gateway 구성을 불러와 확인합니다.

In [ ]:
# Gateway 구성 불러오기
gateway_config_file = "config.json"

with open(gateway_config_file, "r") as f:
    gateway_config = json.load(f)

print("✅ Gateway Configuration Loaded\n")
print("=" * 60)
print(f"Gateway ID:  {gateway_config['gateway']['gateway_id']}")
print(f"Gateway ARN: {gateway_config['gateway']['gateway_arn']}")
print(f"Gateway URL: {gateway_config['gateway']['gateway_url']}")
print(f"Region:      {gateway_config['region']}")
print("\nOAuth Configuration:")
print(f"  Client ID:  {gateway_config['gateway']['client_info']['client_id']}")
print(f"  Token URL:  {gateway_config['gateway']['client_info']['token_endpoint']}")
print("=" * 60)

# 나중에 사용할 수 있도록 저장
GATEWAY_ARN = gateway_config["gateway"]["gateway_arn"]
GATEWAY_ID = gateway_config["gateway"]["gateway_id"]
GATEWAY_URL = gateway_config["gateway"]["gateway_url"]

### Gateway에 생성된 도구를 사용하는 Agent 실행

In [ ]:
# Agent 세션 가져오기
from scripts.agent_with_tools import AgentSession

# 컨텍스트 관리자에서 Agent 사용(설정 및 정리를 자동으로 처리)
with AgentSession() as session:
    # 설정 중에 Agent가 사용 가능한 모든 도구를 나열

    # 이제 다양한 prompt로 Agent를 호출할 수 있음
    response1 = session.invoke("What tools do you have access to?")

    response2 = session.invoke("Create an application for US region with $5M coverage")

    response3 = session.invoke(
        "Invoke the risk model with public API classification and data governance approval set to true"
    )

    response4 = session.invoke("Approve underwriting for $75000 claim with medium risk level")

# 'with' 블록을 벗어나면 세션이 자동으로 정리됨
print("=" * 60)
print(f"🚀 The agent has access to the following tools configured on the Gateway: {response1}\n")
print(f"🚀 The agent can create applications without any limits: {response2}\n")
print(f"🚀 The agent can invoke the risk model: {response3}\n")
print(f"🚀 The agent is able to approve the insurance claims: {response4}\n")
print("=" * 60)

---

# Step 3: Policy Engine 및 정책 생성

이제 Gateway의 도구에 대한 액세스를 제어하도록 Cedar 정책을 포함하는 Policy Engine을 생성합니다.

## Policy Engine이란?

Policy Engine은 요청을 Cedar 정책에 따라 실시간으로 평가합니다. 다음 두 가지 모드로 작동합니다.
- **LOG_ONLY**: 평가하지만 차단하지 않음(테스트용)
- **ENFORCE**: 규정을 준수하지 않는 요청을 적극적으로 차단함(프로덕션용)

Gateway가 Policy Engine에 연결되면 특정 정책에서 액세스를 허용하지 않는 한 기본 동작은 거부입니다. 빈 Policy Engine에서는 Gateway의 어떤 도구에도 액세스할 수 없습니다. 

### 암호화 키

Policy Engine을 생성할 때 AWS Key Management Service(KMS)에 있는 고객 소유 및 관리 키인 Customer Managed Key(CMK)를 적용할 수 있습니다. Policy Engine을 생성하기 전에 CMK의 ARN을 입력하라는 메시지가 표시됩니다. CMK를 사용하지 않으려면 값을 입력하지 않고 넘어가면 됩니다.

In [ ]:
# 사용자에게 CMK ARN 입력 요청
user_input = input("Please enter the CMK ARN.  To skip enter a blank value: ")

# 사용자가 값을 입력했으면 사용하고, 그렇지 않으면 None 전달
cmk_arn = None
if user_input.strip():  # 입력값이 비어 있지 않은지 확인
    cmk_arn = user_input.strip()

### Policy Engine 생성

먼저 Cedar 정책을 저장할 Policy Engine을 생성합니다.

In [ ]:
# AgentCore Starter Toolkit에서 PolicyClient 가져오기
import boto3

# Policy Engine 생성에 필요한 구성 요소 준비
client = boto3.client("bedrock-agentcore-control", region_name=region)

# Policy Engine 생성
print("🔧 Creating Policy Engine...")

# 클라이언트로 보낼 데이터 구성
request = {
    "name": "InsurancePolicyEngine",
    "description": "Policy engine for insurance underwriting governance",
}

# CMK가 제공되었으면 Policy Engine 생성 요청에 추가
if cmk_arn:
    request["encryptionKeyArn"] = cmk_arn


# Policy Engine 생성(동일한 이름으로 이미 존재하면 재사용)
try:
    engine = client.create_policy_engine(**request)
except client.exceptions.ConflictException:
    existing = client.list_policy_engines()
    engine = next(e for e in existing.get("policyEngines", []) if e["name"] == request["name"])

print(f"✓ Policy Engine: {engine['policyEngineId']}\n")

In [ ]:
# 구성 파일에 Policy Engine 저장
with open("config.json", "r") as f:
    config = json.load(f)

# 기존 데이터를 제거하지 않고 Policy Engine 정보 추가
config["policy_engine_id"] = engine["policyEngineId"]
config["policy_engine_arn"] = engine["policyEngineArn"]

# 업데이트된 구성을 다시 기록
with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

print("✅ Policy engine information added to config.json")

In [ ]:
# 연결하기 전에 Policy Engine이 ACTIVE 상태가 될 때까지 대기
import time

_pe_client = boto3.client("bedrock-agentcore-control", region_name=region)
for _ in range(60):
    _pe_status = _pe_client.get_policy_engine(policyEngineId=engine["policyEngineId"]).get("status")
    if _pe_status == "ACTIVE":
        break
    print(f"Policy Engine status: {_pe_status}, waiting...")
    time.sleep(5)
print(f"Policy Engine is {_pe_status}")

# Gateway에 Policy Engine 연결
from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient

gateway_client = GatewayClient(region_name=region)
gateway_client.logger.setLevel(logging.INFO)

gateway_client.update_gateway_policy_engine(
    gateway_identifier=config["gateway"]["gateway_id"],
    policy_engine_arn=engine["policyEngineArn"],
    mode="ENFORCE",
)
print("✓ Policy Engine attached to Gateway\n")

### Policy Engine이 Gateway에 연결되었으므로 이제 도구 목록 API 호출과 Agent의 Gateway를 통한 액세스 모두에서 기본적으로 도구가 차단됩니다.

In [ ]:
# Agent 세션 가져오기
from scripts.agent_with_tools import list_available_tools, fetch_access_token

client_info = config["gateway"]["client_info"]

CLIENT_ID = client_info["client_id"]
CLIENT_SECRET = client_info["client_secret"]
TOKEN_URL = client_info["token_endpoint"]
access_token = fetch_access_token(CLIENT_ID, CLIENT_SECRET, TOKEN_URL)

print("=" * 60)
print("🚀 The agent has access to the following tools configured on the Gateway: \n")
print(list_available_tools(config["gateway"]["gateway_url"], access_token))

### Cedar 정책 생성

이제 보장 금액이 100만 달러 이하인 경우 보험 신청 생성을 허용하는 Cedar 정책을 생성합니다.

## 정책 규칙

```cedar
permit(
  principal,
  action == AgentCore::Action::"ApplicationToolTarget___create_application",
  resource == AgentCore::Gateway::"<gateway-arn>"
) when {
  context.input.coverage_amount <= 1000000
};
```

이 정책의 의미는 다음과 같습니다.
- ✅ 100만 달러 이하의 보험 신청 생성: **ALLOWED**
- ❌ 100만 달러를 초과하는 보험 신청 생성: **DENIED**

In [ ]:
from bedrock_agentcore_starter_toolkit.operations.policy.client import PolicyClient

# Policy 클라이언트 생성
policy_client = PolicyClient(region_name=region)
policy_client.logger.setLevel(logging.INFO)

# Cedar 정책 생성
print("\n📝 Creating Cedar Policy...")
print(f"   Policy Engine ID: {engine['policyEngineArn']}")
GATEWAY_ARN = config["gateway"]["gateway_arn"]

# Cedar 정책문 정의
cedar_statement = (
    f"permit(principal, "
    f'action == AgentCore::Action::"ApplicationToolTarget___create_application", '
    f'resource == AgentCore::Gateway::"{GATEWAY_ARN}") '
    f"when {{ context.input.coverage_amount <= 1000000 }};"
)

try:
    policy = policy_client.create_or_get_policy(
        policy_engine_id=engine["policyEngineId"],
        name="create_application_policy",
        description="Allow application creation under $1M",
        definition={"cedar": {"statement": cedar_statement}},
    )
    print(f"✓ Policy: {policy['policyId']}\n")
except Exception as e:
    print(f"⚠️  Policy creation failed: {e}")
    print("   This may be due to Cedar validation findings (e.g., policy too restrictive or schema issues).")
    print("   Retrying with validation_mode='IGNORE_ALL_FINDINGS'...")
    policy = policy_client.create_or_get_policy(
        policy_engine_id=engine["policyEngineId"],
        name="create_application_policy",
        description="Allow application creation under $1M",
        definition={"cedar": {"statement": cedar_statement}},
        validation_mode="IGNORE_ALL_FINDINGS",
    )
    print(f"✓ Policy created with IGNORE_ALL_FINDINGS: {policy['policyId']}\n")

# 구성에 저장
config["policy_id"] = policy["policyId"]
with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

---

# Step 4: AI Agent로 정책 적용 테스트

이제 실제 AI Agent로 정책을 테스트해 보겠습니다!
`create_application` 도구에 연결한 정책 덕분에 이제 Gateway가 이 도구를 목록에 표시하고 Agent가 호출할 수 있습니다.

## 테스트 시나리오

다음 두 가지 시나리오를 테스트합니다.

### 테스트 1: ALLOWED 시나리오 ✅
- **요청**: 보장 금액이 75만 달러인 신청 생성
- **예상 결과**: Policy가 허용하고 Lambda가 실행되어 신청이 생성됨
- **이유**: $750K <= $1M(정책 한도 이내)


In [ ]:
# 컨텍스트 관리자에서 Agent 사용(설정 및 정리를 자동으로 처리)
with AgentSession() as session:
    # 설정 중에 Agent가 사용 가능한 모든 도구를 나열

    # 이제 다양한 prompt로 Agent를 호출할 수 있음
    response1 = session.invoke("What tools do you have access to?")

    response2 = session.invoke("Create an application for US region with $750,000 coverage")

### 테스트 2: DENIED 시나리오 ❌
- **요청**: 보장 금액이 150만 달러인 신청 생성
- **예상 결과**: Policy가 차단하고 Lambda는 실행되지 않음
- **이유**: $1.5M > $1M(정책 한도 초과)

In [ ]:
with AgentSession() as session:
    # 설정 중에 Agent가 사용 가능한 모든 도구를 나열

    response2 = session.invoke("Create an application for US region with $1.5M coverage")

## 리소스 정리

<div style="background-color: #d1ecf1; border-left: 4px solid #0c5460; padding: 10px; margin: 10px 0; color: #000;">
    <strong style="color: #000;">ℹ️ 참고:</strong> 02-Natural-Language-Policy-Authoring/NL-Authoring-Policy.ipynb에서는 이 데모에서 설정한 Gateway를 재사용합니다. 지금은 리소스 정리 단계를 건너뛰고 두 번째 실습에서 NL2Cedar 기능을 테스트한 후 수행해도 됩니다. 
</div>


Policy Engine과 정책 리소스는 다음 순서로 정리합니다.
1. `update_gateway` CLI에 빈 Policy Engine을 전달하여 Gateway와 Policy Engine 간 연결을 삭제합니다.

2. Policy Engine의 모든 정책을 삭제합니다.

3. Policy Engine을 삭제합니다.

In [ ]:
from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient
from bedrock_agentcore_starter_toolkit.operations.policy.client import PolicyClient

with open("config.json", "r") as f:
    config = json.load(f)

# 먼저 Policy Engine 정리
print("🧹 Cleaning up Policy Engine...")
policy_client = PolicyClient(region_name=config["region"])
policy_client.cleanup_policy_engine(config["policy_engine_id"])
print("✓ Policy Engine cleaned up\n")

# 그런 다음 Gateway 정리
print("🧹 Cleaning up Gateway...")
gateway_client = GatewayClient(region_name=config["region"])
gateway_client.cleanup_gateway(config["gateway"]["gateway_id"], config["gateway"]["client_info"])
print("✅ Cleanup complete!")